In [1]:
import google.auth
import numpy as np
import pandas as pd
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [29]:

import gc
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from pyproj import Transformer

In [3]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [4]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

In [5]:
@cache
def gcs_pandas():
    return GCSPandas()

In [6]:
from functools import cache

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [7]:
gcsgp = GCSGeoPandas()

In [8]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [9]:
import rasterio
import rasterio.features
import rasterio.warp

from rasterio.plot import show
import matplotlib.pyplot as plt


In [10]:
conveyal_data = rasterio.open('CentralCal_TRANSIT_BASELINE_CSISWeight_P50_C90.tif')

In [11]:
conveyal_data

<open DatasetReader name='CentralCal_TRANSIT_BASELINE_CSISWeight_P50_C90.tif' mode='r'>

In [13]:
conveyal_df_metadata = conveyal_data.read(1)

In [23]:
# Convert this to a dataframe
values = conveyal_data.read(1)

In [24]:
rows, cols = np.indices(values.shape)

In [26]:

xs, ys = rasterio.transform.xy(
        conveyal_data.transform,
        rows.flatten(),
        cols.flatten()
    )

raster_df = pd.DataFrame({
        "row": rows.flatten(),
        "column": cols.flatten(),
        "x": xs,
        "y": ys,
        "accessible_jobs": values.flatten()
    })


In [30]:

transformer = Transformer.from_crs(
    conveyal_data.crs,
    "EPSG:4326",
    always_xy=True
)

raster_df["lon"], raster_df["lat"] = transformer.transform(
    raster_df["x"].values,
    raster_df["y"].values
)


In [41]:
raster_df.accessible_jobs.describe()

count   3985856.00
mean         75.45
std         755.09
min           0.00
25%           0.00
50%           0.00
75%           0.00
max       18800.00
Name: accessible_jobs, dtype: float64

In [33]:
len(raster_df)

3985856

In [43]:
len(raster_df.loc[raster_df.accessible_jobs > 0])

228305

In [50]:
raster_df.loc[raster_df.accessible_jobs > 0].accessible_jobs.describe()

count   228305.00
mean      1317.21
std       2884.17
min          1.00
25%          1.00
50%          5.00
75%        878.00
max      18800.00
Name: accessible_jobs, dtype: float64

In [32]:
raster_df.x.nunique()

2296

In [34]:
raster_df.y.nunique()

1736

In [55]:
raster_df = gpd.GeoDataFrame(
    raster_df, geometry=gpd.points_from_xy(raster_df.lon, raster_df.lat))



In [53]:
# Why are there so many duplicates
# raster_df.loc[raster_df.accessible_jobs > 5000].sample(100).explore("accessible_jobs")

* From that cell's location, approximately 18,800 jobs are reachable by transit under the scenario and time threshold used to create the raster.
* if: the metric is "jobs accessible within 60 minutes by transit," and a particular pixel has the value 18,800, then a person starting from that pixel could potentially reach about 18,800 jobs within 60 minutes by transit.
* Think of it like a heat map

In [17]:
census_block_url = "gs://calitp-analytics-data/data-analyses/equity_index/unbuffered_censusblocks_2020/combined_unbufferd_ca_2020.parquet"

In [56]:
census_df = to_snakecase(gcsgp.read_parquet(
   census_block_url)).to_crs(raster_df.crs)

In [19]:
census_df.crs

<Projected CRS: EPSG:3857>
Name: WGS 84 / Pseudo-Mercator
Axis Info [cartesian]:
- [east]: Easting (metre)
- [north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Popular Visualisation Pseudo Mercator
Datum: World Geodetic System 1984
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [57]:
join1 = gpd.sjoin(
    raster_df,
    census_df,
    how="inner",
    predicate="within"
)


In [58]:
join1.head(1)

,row,column,x,y,accessible_jobs,lon,lat,geometry,index_right,statefp20,countyfp20,tractce20,blockce20,geoid20,name20,mtfcc20,ur20,uace20,uatype20,funcstat20,aland20,awater20,intptlat20,intptlon20,housing20,pop20,county_name
0,0,0,-13785418.05,4813851.17,0.00,-123.84,39.64,POINT (-123.83652 39.64060),215295,06,045,990100,0002,060459901000002,Block 0002,G5040,R,None,None,S,0,116224734,+39.6471455,-123.8310518,0,0,Mendocino


In [75]:
agg1 = (
    join1
    .groupby(["geoid20","county_name"])
    .agg(
        avg_accessible_jobs=("accessible_jobs", "mean"),
        max_accessible_jobs=("accessible_jobs", "max"),
        pixel_count=("accessible_jobs", "size")
    )
    .reset_index()
)


In [71]:
census_df.columns

Index(['statefp20', 'countyfp20', 'tractce20', 'blockce20', 'geoid20',
       'name20', 'mtfcc20', 'ur20', 'uace20', 'uatype20', 'funcstat20',
       'aland20', 'awater20', 'intptlat20', 'intptlon20', 'housing20', 'pop20',
       'geometry', 'county_name'],
      dtype='object')

In [76]:
agg1 = pd.merge( census_df[["geoid20","geometry"]], agg1, on = ["geoid20"])

In [ ]:
agg1.explore("max_accessible_jobs")

In [64]:
agg1.county_name.value_counts()

county_name
Fresno           9947
Sacramento       8709
Santa Clara      8482
Alameda          6512
Tulare           5927
Contra Costa     5748
San Joaquin      5314
Sonoma           4336
Stanislaus       4334
Placer           3827
Monterey         3825
Solano           3653
Merced           3615
San Mateo        3498
El Dorado        3057
Madera           2750
Mendocino        2497
Kings            2128
Nevada           2100
Tuolumne         2081
Lake             1904
Marin            1790
Yolo             1770
Santa Cruz       1743
San Francisco    1644
Butte            1559
Inyo             1504
Sutter           1471
Calaveras        1464
Yuba             1179
Mono             1131
Napa             1067
Colusa           1030
Mariposa          937
Amador            906
San Benito        865
Glenn             646
Sierra            625
Alpine            179
Plumas             16
Name: count, dtype: int64